In [1]:
import os
import asyncio
from typing import List, Dict, Any
from dotenv import load_dotenv


In [2]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console

In [3]:
load_dotenv()

True

In [4]:
os.getenv("OPENAI_MODEL_ID", "coding-glm-5.1-free")

'gpt-4.1-free'

In [12]:
def create_openai_model_client():
    """创建 OpenAI 模型客户端用于测试"""
    return OpenAIChatCompletionClient(
        model=os.getenv("OPENAI_MODEL_ID", "coding-glm-5.1-free"),
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL", "https://aihubmix.com/v1"),
        # 核心调整：显式指定模型的能力特征
        model_info={
            "vision": False,
            "function_calling": True,  # 设为 True 才能在 Agent 中使用 Tools
            "json_output": True,       # 根据模型实际是否支持决定
            "family": "unknown",       # 可使用 "unknown" 或其他自定义名称
            "structured_output": True
        }
    )

def create_product_manager(model_client):
    """创建产品经理智能体"""
    system_message = """你是一位经验丰富的产品经理，专门负责软件产品的需求分析和项目规划。

你的核心职责包括：
1. **需求分析**：深入理解用户需求，识别核心功能和边界条件
2. **技术规划**：基于需求制定清晰的技术实现路径
3. **风险评估**：识别潜在的技术风险和用户体验问题
4. **协调沟通**：与工程师和其他团队成员进行有效沟通

当接到开发任务时，请按以下结构进行分析：
1. 需求理解与分析
2. 功能模块划分
3. 技术选型建议
4. 实现优先级排序
5. 验收标准定义

请简洁明了地回应，并在分析完成后说"请工程师开始实现"。"""

    return AssistantAgent(
        name="ProductManager",
        model_client=model_client,
        system_message=system_message,
    )

def create_engineer(model_client):
    """创建软件工程师智能体"""
    system_message = """你是一位资深的软件工程师，擅长 Python 开发和 Web 应用构建。

你的技术专长包括：
1. **Python 编程**：熟练掌握 Python 语法和最佳实践
2. **Web 开发**：精通 Streamlit、Flask、Django 等框架
3. **API 集成**：有丰富的第三方 API 集成经验
4. **错误处理**：注重代码的健壮性和异常处理

当收到开发任务时，请：
1. 仔细分析技术需求
2. 选择合适的技术方案
3. 编写完整的代码实现
4. 添加必要的注释和说明
5. 考虑边界情况和异常处理

请提供完整的可运行代码，并在完成后说"请代码审查员检查"。"""

    return AssistantAgent(
        name="Engineer",
        model_client=model_client,
        system_message=system_message,
    )

def create_code_reviewer(model_client):
    """创建代码审查员智能体"""
    system_message = """你是一位经验丰富的代码审查专家，专注于代码质量和最佳实践。

你的审查重点包括：
1. **代码质量**：检查代码的可读性、可维护性和性能
2. **安全性**：识别潜在的安全漏洞和风险点
3. **最佳实践**：确保代码遵循行业标准和最佳实践
4. **错误处理**：验证异常处理的完整性和合理性

审查流程：
1. 仔细阅读和理解代码逻辑
2. 检查代码规范和最佳实践
3. 识别潜在问题和改进点
4. 提供具体的修改建议
5. 评估代码的整体质量

请提供具体的审查意见，完成后说"代码审查完成，请用户代理测试"。"""

    return AssistantAgent(
        name="CodeReviewer",
        model_client=model_client,
        system_message=system_message,
    )

def create_user_proxy():
    """创建用户代理智能体"""
    return UserProxyAgent(
        name="UserProxy",
        description="""用户代理，负责以下职责：
1. 代表用户提出开发需求
2. 执行最终的代码实现
3. 验证功能是否符合预期
4. 提供用户反馈和建议

完成测试后请回复 TERMINATE。""",
    )

In [13]:
async def run_software_development_team():
    print("🔧 正在初始化模型客户端...")
    model_client = create_openai_model_client()

    print("👥 正在创建智能体团队...")
    product_manager = create_product_manager(model_client)
    engineer = create_engineer(model_client)
    code_reviewer = create_code_reviewer(model_client)
    user_proxy = create_user_proxy()

    termination = TextMentionTermination("TERMINATE")

    team_chat = RoundRobinGroupChat(
        participants=[
            product_manager,
            engineer,
            code_reviewer,
            user_proxy
        ],
        termination_condition=termination,
        max_turns=20,
    )

    task = """我们需要开发一个比特币价格显示应用，具体要求如下：
    核心功能：
    - 试试显示比特币当前价格（USD）
    - 显示24小时价格变化趋势（涨跌幅和涨跌额）
    - 提供价格刷新功能

    技术要求：
    - 使用Streamlit框架创建Web应用
    - 界面简洁美观，用户友好
    - 添加适当的错误处理和加载状态
    
    请团队协作完成这个任务，从需求分析到最终实现。
    """

    print("🚀 启动 AutoGen 软件开发团队协作...")
    print("=" * 60)

    result = await Console(team_chat.run_stream(task=task))

    print("\n" + "=" * 60)
    print("✅ 团队协作完成！")

    return result

In [14]:
try:
    result = await run_software_development_team()
    
    print(f"\n📋 协作结果摘要：")
    print(f"- 参与智能体数量：4个")
    print(f"- 任务完成状态：{'成功' if result else '需要进一步处理'}")
    
except ValueError as e:
    print(f"❌ 配置错误：{e}")
    print("请检查 .env 文件中的配置是否正确")
except Exception as e:
    print(f"❌ 运行错误：{e}")
    import traceback
    traceback.print_exc()

🔧 正在初始化模型客户端...
👥 正在创建智能体团队...
🚀 启动 AutoGen 软件开发团队协作...
---------- TextMessage (user) ----------
我们需要开发一个比特币价格显示应用，具体要求如下：
    核心功能：
    - 试试显示比特币当前价格（USD）
    - 显示24小时价格变化趋势（涨跌幅和涨跌额）
    - 提供价格刷新功能

    技术要求：
    - 使用Streamlit框架创建Web应用
    - 界面简洁美观，用户友好
    - 添加适当的错误处理和加载状态
    
    请团队协作完成这个任务，从需求分析到最终实现。
    
---------- TextMessage (ProductManager) ----------
1. 需求理解与分析

- 用户需要一个可在网页端快速查看比特币当前美元价格的工具。
- 用户希望直观获取24小时内比特币的涨跌幅和涨跌额，便于判断行情走势。
- 用户可以主动刷新价格数据，确保信息及时。
- 需要适应常见网络波动或API异常等情况，保持良好用户体验。
- 产品需基于Streamlit开发，追求简单易用，同时界面美观。

2. 功能模块划分

- 价格数据获取模块
    - 调用公开比特币行情API（如CoinGecko、CoinMarketCap等），定期和手动刷新数据  
    - 获取当前价格、24小时内价格变动百分比及绝对额
- 数据展示模块
    - 主界面显示当前价格
    - 辅助展示涨跌百分比及金额（红绿标示）
    - 展示数据最后一次的更新时间
- 交互模块
    - “刷新”按钮，触发即时获取最新数据
    - 加载时显示loading动画，异常时显示错误提示
- 美观及易用性模块
    - 简洁布局，颜色区分涨跌
    - 响应式适配主流屏幕

3. 技术选型建议

- 前端/框架：Streamlit（满足交互展示和快速开发的要求）
- 数据源：CoinGecko API（免费、无需授权、稳定）
- 开发语言：Python 3
- 库支持：requests（API请求）、pandas（数据处理，根据应用复杂度选用）、Streamlit自带UI与异步特性
- 错误处理：try-except捕获A

/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_agentchat/agents/_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-4.1-free != gpt-4.1-2025-04-14. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4.1-2025-04-14 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- TextMessage (Engineer) ----------
收到需求，下面将按照规范逐步实现。团队成员分工如下：

---

### 1. 需求分析与技术方案拆解

#### 主要API
- 选用 [CoinGecko API](https://www.coingecko.com/en/api)，其免费且无需API key。
- 价格接口示例：https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd&include_24hr_change=true

API 返回样例（精简）：
```json
{
  "bitcoin": {
    "usd": 67654,
    "usd_24h_change": 2.91635
  }
}
```
> 说明：直接返回当前价格（usd）与24小时涨跌百分比（usd_24h_change），但不返回绝对涨跌额。需结合api/v3/coins/bitcoin/history接口或简单倒推出绝对涨跌额。
- 也可拉取 `api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=1` 获取24小时内价格列表。

---

### 2. 设计概要

- **数据层**  
  - 拉取当前价格和24小时涨跌 %
  - 拉取24小时前的价格，计算绝对涨跌额
- **展示层**  
  - Streamlit主页面展示
    - 比特币标志与名称
    - 当前价格，24小时涨跌额/幅度，颜色区分
    - 刷新按钮，最后更新时间
    - 加载动画/异常提示
- **交互层**  
  - 刷新按钮触发数据更新
  - 自动页面加载数据

---

### 3. 代码实现

**代码主要逻辑：**
- 函数 get_bitcoin_data() 完成API请求和数据处理
- 主页面streamlit应用负责UI渲染

---

```python
# btc_price_app.py

import requests
import streamlit as st
from datetime import datetime
from typing impo

Error processing publish message for Engineer_28b273c3-6370-42f9-95ef-e79b17cec57c/28b273c3-6370-42f9-95ef-e79b17cec57c
Traceback (most recent call last):
  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_core/_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_core/_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_agentchat/teams/_group_chat/_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_core/_routed_agent.py", line 485, in on_message_impl
    return await h(self, message, ctx)
  File "/User

❌ 运行错误：RateLimitError: Error code: 429 - {'error': {'message': 'Sorry, you have reached the limit of the free model quota. Please switch to a paid model to enjoy unlimited concurrency. https://console.aihubmix.com/topup (tid: 2026050700431854579439021869601)', 'type': 'Aihubmix_api_error'}}
Traceback:
Traceback (most recent call last):

  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_agentchat/teams/_group_chat/_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):

  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_agentchat/agents/_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(

  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_agentchat/agents/_assistant_agent.py", line 1109, in _ca

Traceback (most recent call last):
  File "/var/folders/vb/9nz8kb0j30v3kzrqkxc6hvhm0000gn/T/ipykernel_4885/401421352.py", line 2, in <module>
    result = await run_software_development_team()
  File "/var/folders/vb/9nz8kb0j30v3kzrqkxc6hvhm0000gn/T/ipykernel_4885/2273246474.py", line 41, in run_software_development_team
    result = await Console(team_chat.run_stream(task=task))
  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_agentchat/ui/_console.py", line 117, in Console
    async for message in stream:
  File "/Users/yuhangwang/Desktop/Proj/hello-agents-learn/venv/lib/python3.10/site-packages/autogen_agentchat/teams/_group_chat/_base_group_chat.py", line 554, in run_stream
    raise RuntimeError(str(message.error))
RuntimeError: RateLimitError: Error code: 429 - {'error': {'message': 'Sorry, you have reached the limit of the free model quota. Please switch to a paid model to enjoy unlimited concurrency. https://console.aihubmix.co